# Text to SQL Decoder-only Baseline

Decoder-only experiment for Text to SQL task.

This notebook uses a causal language model. Instead of encoder-decoder generation, we format the task as prompt completion:

```text
prompt -> SQL
```

## 1. Install

In [ ]:
!pip -q install -U transformers accelerate sentencepiece scikit-learn
!pip -q install "datasets<4.0.0" "pandas==2.2.2"

## 2. Imports

In [ ]:
import random
import re
import unicodedata

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)

## 3. Config

In [ ]:
MODEL_NAME = "gpt2"
DATASET_NAME = "Salesforce/wikisql"

BASELINE_T5_SMALL_EM = 0.594
BASELINE_T5_BASE_LORA_EM = 0.605

USE_FULL_TRAIN = True
TRAIN_SIZE = 10000

VALID_SIZE = 1000
TEST_SIZE = 1000

MAX_LENGTH = 384

BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8
EPOCHS = 2
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
SEED = 42

OUTPUT_DIR = "./runs/gpt2-full-finetune"

## 4. Setup

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

if device == "cuda":
    print(torch.cuda.get_device_name(0))

## 5. Load WikiSQL

In [ ]:
dataset = load_dataset(DATASET_NAME, trust_remote_code=True)
dataset

## 6. WikiSQL serialization

In [ ]:
AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<", "OP"]


def clean_text(x):
    x = str(x).replace("\n", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def get_columns(example):
    table = example["table"]

    if "header" in table:
        return table["header"]
    if "headers" in table:
        return table["headers"]
    if "column_names" in table:
        return table["column_names"]

    raise ValueError(f"Unknown table format: {table.keys()}")


def iter_conditions(conditions):
    if isinstance(conditions, dict):
        col_indices = conditions.get("column_index", [])
        op_indices = conditions.get("operator_index", [])
        values = conditions.get("condition", [])

        for col_idx, op_idx, value in zip(col_indices, op_indices, values):
            yield col_idx, op_idx, value
    else:
        for cond in conditions:
            yield cond[0], cond[1], cond[2]


def serialize_sql(example):
    sql = example["sql"]
    columns = get_columns(example)

    select_col = clean_text(columns[sql["sel"]])
    agg = AGG_OPS[sql["agg"]]

    if agg:
        query = f"SELECT {agg}({select_col}) FROM table"
    else:
        query = f"SELECT {select_col} FROM table"

    where_parts = []

    for col_idx, op_idx, value in iter_conditions(sql.get("conds", [])):
        col_name = clean_text(columns[col_idx])
        op = COND_OPS[op_idx] if op_idx < len(COND_OPS) else "="
        value = clean_text(value)
        where_parts.append(f"{col_name} {op} '{value}'")

    if where_parts:
        query += " WHERE " + " AND ".join(where_parts)

    return query


def make_prompt(example):
    question = clean_text(example["question"])
    columns = " | ".join(clean_text(c) for c in get_columns(example))

    return (
        "Task: translate question to SQL.\n"
        f"Question: {question}\n"
        f"Columns: {columns}\n"
        "SQL:"
    )


def make_full_text(example):
    return make_prompt(example) + " " + serialize_sql(example)

## 7. Check prompt format

In [ ]:
for i in range(3):
    ex = dataset["train"][i]
    print(make_full_text(ex))
    print("-" * 80)

## 8. Prepare data

In [ ]:
if USE_FULL_TRAIN:
    train_raw = dataset["train"].shuffle(seed=SEED)
else:
    train_raw = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))

valid_raw = dataset["validation"].shuffle(seed=SEED).select(range(VALID_SIZE))
test_raw = dataset["test"].shuffle(seed=SEED).select(range(TEST_SIZE))

print("train:", len(train_raw))
print("valid:", len(valid_raw))
print("test:", len(test_raw))

## 9. Load tokenizer and model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

## 10. Tokenize with prompt masking

In [ ]:
def tokenize_example(example):
    prompt = make_prompt(example)
    target = " " + serialize_sql(example) + tokenizer.eos_token
    full_text = prompt + target

    full = tokenizer(
        full_text,
        max_length=MAX_LENGTH,
        truncation=True
    )

    prompt_ids = tokenizer(
        prompt,
        max_length=MAX_LENGTH,
        truncation=True
    )["input_ids"]

    labels = full["input_ids"].copy()
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    full["labels"] = labels
    return full


train_ds = train_raw.map(tokenize_example, remove_columns=train_raw.column_names)
valid_ds = valid_raw.map(tokenize_example, remove_columns=valid_raw.column_names)
test_ds = test_raw.map(tokenize_example, remove_columns=test_raw.column_names)

## 11. Data collator

In [ ]:
class CausalText2SQLCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        input_ids = []
        attention_mask = []
        labels = []

        for f in features:
            pad_len = max_len - len(f["input_ids"])

            input_ids.append(f["input_ids"] + [self.tokenizer.pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


data_collator = CausalText2SQLCollator(tokenizer)

## 12. Train

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator
)

trainer.train()

## 13. Generation

In [ ]:
def extract_sql(generated_text):
    if "SQL:" in generated_text:
        sql = generated_text.split("SQL:", 1)[1]
    else:
        sql = generated_text

    sql = sql.strip()

    stop_markers = ["\nTask:", "\nQuestion:", "\nColumns:"]
    for marker in stop_markers:
        if marker in sql:
            sql = sql.split(marker, 1)[0].strip()

    return sql


def generate_sql(example, max_new_tokens=96):
    prompt = make_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return extract_sql(generated)


for i in range(5):
    ex = test_raw[i]
    target = serialize_sql(ex)
    pred = generate_sql(ex)

    print("PROMPT:", make_prompt(ex))
    print("TARGET:", target)
    print("PRED:", pred)
    print("-" * 80)

## 14. Metrics

In [ ]:
def normalize_sql(sql):
    sql = unicodedata.normalize("NFKC", str(sql))
    sql = sql.lower()
    sql = sql.replace('"', "'")
    sql = re.sub(r"\s+", " ", sql)
    sql = re.sub(r"\s*([(),=<>])\s*", r"\1", sql)
    return sql.strip()


def parse_sql_parts(sql):
    sql = normalize_sql(sql)
    match = re.match(r"^select (.*?) from table(?: where (.*))?$", sql)

    if not match:
        return {
            "valid": False,
            "select": "",
            "where": "",
            "agg": ""
        }

    select_part = match.group(1).strip()
    where_part = match.group(2).strip() if match.group(2) else ""

    agg_match = re.match(r"^(max|min|count|sum|avg)\((.*)\)$", select_part)
    agg = agg_match.group(1) if agg_match else ""

    return {
        "valid": True,
        "select": select_part,
        "where": where_part,
        "agg": agg
    }


def compute_metrics_table(preds, refs):
    pred_parts = [parse_sql_parts(p) for p in preds]
    ref_parts = [parse_sql_parts(r) for r in refs]

    exact = [
        normalize_sql(p) == normalize_sql(r)
        for p, r in zip(preds, refs)
    ]

    valid = [p["valid"] for p in pred_parts]

    select_match = [
        p["select"] == r["select"]
        for p, r in zip(pred_parts, ref_parts)
    ]

    where_match = [
        p["where"] == r["where"]
        for p, r in zip(pred_parts, ref_parts)
    ]

    agg_match = [
        p["agg"] == r["agg"]
        for p, r in zip(pred_parts, ref_parts)
    ]

    return pd.DataFrame([
        {
            "model": "t5-small-full",
            "normalized_exact_match": BASELINE_T5_SMALL_EM,
            "valid_sql_like": None,
            "select_match": None,
            "where_match": None,
            "aggregation_match": None
        },
        {
            "model": "t5-base-lora",
            "normalized_exact_match": BASELINE_T5_BASE_LORA_EM,
            "valid_sql_like": 0.714,
            "select_match": 0.912,
            "where_match": 0.756,
            "aggregation_match": 1.000
        },
        {
            "model": "gpt2-full",
            "normalized_exact_match": float(np.mean(exact)),
            "valid_sql_like": float(np.mean(valid)),
            "select_match": float(np.mean(select_match)),
            "where_match": float(np.mean(where_match)),
            "aggregation_match": float(np.mean(agg_match))
        }
    ])

## 15. Evaluate

In [ ]:
test_targets = [serialize_sql(ex) for ex in test_raw]
test_preds = [generate_sql(ex) for ex in test_raw]

metrics_df = compute_metrics_table(test_preds, test_targets)
metrics_df

## 16. Error analysis

In [ ]:
results = pd.DataFrame({
    "prompt": [make_prompt(ex) for ex in test_raw],
    "target": test_targets,
    "prediction": test_preds
})

results["target_norm"] = results["target"].apply(normalize_sql)
results["prediction_norm"] = results["prediction"].apply(normalize_sql)
results["correct"] = results["target_norm"] == results["prediction_norm"]

results["correct"].value_counts(normalize=True)

In [ ]:
results[results["correct"] == False][["prompt", "target", "prediction"]].head(20)

## 17. Save model

In [ ]:
save_dir = "./gpt2-text2sql-full-finetune"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

save_dir

## 18. Custom examples

In [ ]:
def predict_custom(question, columns):
    fake_example = {
        "question": question,
        "table": {"header": [col.strip() for col in columns.split("|")]},
        "sql": {"sel": 0, "agg": 0, "conds": []}
    }

    return generate_sql(fake_example)


examples = [
    ("What is the highest score?", "player | team | score | year"),
    ("How many players are from France?", "player | country | age | team"),
    ("What team has score 10?", "team | score | year")
]

for question, columns in examples:
    print("QUESTION:", question)
    print("COLUMNS:", columns)
    print("PRED:", predict_custom(question, columns))
    print("-" * 80)